# ISRO Hackathon: Kaggle Training Pipeline
Welcome to the official Kaggle training notebook for the ISRO Satellite Infrared Colorization project.

### Setup Instructions for Kaggle (Zip Upload Method):
1. On the right-hand panel, go to **Notebook options** -> **Accelerator** and select **TPU VM v3-8** (or GPU T4 x2).
2. Click **Add Data** on the right side and add the zip file you uploaded.
3. Replace `YOUR_DATASET_NAME` below with the exact folder name Kaggle gave your uploaded dataset!
4. Run the cells below to copy the code to a writeable directory and start training!

In [ ]:
# 1. Copy your codebase to the Writeable Working Directory
# ⚠️ REPLACE 'YOUR_DATASET_NAME' WITH THE EXACT FOLDER NAME IN /kaggle/input/ ⚠️
import os
import shutil

KAGGLE_DATASET_NAME = 'YOUR_DATASET_NAME'
INPUT_PATH = f'/kaggle/input/{KAGGLE_DATASET_NAME}'

# Copy the IR-RGB folder from the read-only input to the writeable working directory
!cp -r {INPUT_PATH}/IR-RGB /kaggle/working/IR-RGB

os.chdir('/kaggle/working/IR-RGB')
print("Codebase successfully copied to /kaggle/working/IR-RGB")

In [ ]:
# 2. Fix Configuration File Paths for Kaggle Environment
import yaml

# Kaggle only allows writing to /kaggle/working/
CHECKPOINTS_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Point data paths back to the read-only dataset folder where the images live
# (Make sure these folder names match exactly how they are structured in your zip file)
config['data']['raw_dir'] = f'{INPUT_PATH}/FILES'
config['data']['processed_dir'] = f'{INPUT_PATH}/FILES/processed' 

# Writeable output paths
config['training']['checkpoints_dir'] = CHECKPOINTS_DIR
config['training']['outputs_dir'] = '/kaggle/working/outputs'

with open('configs/config.yaml', 'w') as f:
    yaml.dump(config, f)
print(f"Config paths updated! Checkpoints will save directly to {CHECKPOINTS_DIR}")

In [ ]:
# 3. Install requirements and Kaggle PyTorch XLA
!pip install -r requirements.txt
!pip install torch_xla[tpu]

## Stage 1: Train Real-ESRGAN (Super Resolution)

In [ ]:
# 4. Run the Super-Resolution training script
import os
os.environ['PYTHONPATH'] = '.'
!python training/train_realesrgan.py

## Stage 2: Train Pix2Pix GAN (Colorization)

In [ ]:
# 5. Run the Colorization training script
import os
os.environ['PYTHONPATH'] = '.'
!python training/train_pix2pix.py

## Save Your Results
On Kaggle, everything inside `/kaggle/working/` is saved when you click **Save Version** (Commit). You can download the `.pth` files from the output tab!